In [ ]:
# ============================================
# 0) Imports and module reload
# ============================================
import importlib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import c1_forecasting
import c3_forecasting

importlib.reload(c1_forecasting)
importlib.reload(c3_forecasting)

from c1_forecasting import run_c1_pipeline
from c3_forecasting import run_c3_pipeline


In [ ]:
# ============================================
# 1) Run C1 pipeline
# ============================================
art_c1 = run_c1_pipeline(
    train_path="data/forecasting/train_daily.parquet",
    test_path="data/forecasting/test_daily.parquet",
    cluster_id=1,
    n_periods=4,
    eps_mape=1.0,
    metric_name="bounded_mape",      # reporting metric
    tune=True,
    tuning_objective="wmape",        # optimization objective
    max_trials_per_model_pair=600,   # can increase to 800/1000
    prediction_output_path="forecasting/c1_prediction.parquet",
    random_state=42,
)

print("C1 prediction file:", art_c1.prediction_output_path)
print("\n=== C1 Overall Metrics ===")
display(art_c1.metrics_overall)

print("\n=== C1 Metrics by Period ===")
display(art_c1.metrics_by_period)

if art_c1.tuning_trials is not None:
    print("\n=== C1 Top tuning trials ===")
    display(art_c1.tuning_trials.head(10))

print("\n=== C1 Best Config ===")
print(art_c1.tuning_best_config)

display(art_c1.error_quantiles)

In [ ]:
# ============================================
# 2) Run C3 pipeline
# ============================================
art_c3 = run_c3_pipeline(
    train_path="data/forecasting/train_daily.parquet",
    test_path="data/forecasting/test_daily.parquet",
    cluster_id=3,
    n_periods=4,
    eps_mape=1.0,
    metric_name="bounded_mape",      # reporting metric
    tune=True,
    tuning_objective="wmape",        # optimization objective
    max_trials_per_model_pair=600,   # can increase to 800/1000
    prediction_output_path="forecasting/c3_prediction.parquet",
    random_state=42,
)

print("C3 prediction file:", art_c3.prediction_output_path)
print("\n=== C3 Selected Baseline ===")
print(art_c3.selected_baseline_name)
display(art_c3.baseline_comparison)

print("\n=== C3 Overall Metrics ===")
display(art_c3.metrics_overall)

print("\n=== C3 Metrics by Period ===")
display(art_c3.metrics_by_period)

if art_c3.tuning_trials is not None:
    print("\n=== C3 Top tuning trials ===")
    display(art_c3.tuning_trials.head(10))

print("\n=== C3 Best Config ===")
print(art_c3.tuning_best_config)

display(art_c3.error_quantiles)

In [ ]:
# ============================================
# 3) Assignment-required box plots (error spread by period)
#    - one figure for C1
#    - one figure for C3
# ============================================
# C1
plt.figure(figsize=(12,4))
sns.boxplot(
    data=art_c1.ape_box_df,
    x="period",
    y="APE_EPS_PCT",
    hue="method",
    showfliers=False
)
plt.title("C1 Epsilon-APE Boxplot (No y-limit)")
plt.ylabel("Epsilon APE (%)")
plt.xlabel("Test period")
plt.legend(title="")
plt.tight_layout()
plt.show()

# C3
plt.figure(figsize=(12,4))
sns.boxplot(
    data=art_c3.ape_box_df,
    x="period",
    y="APE_EPS_PCT",
    hue="method",
    showfliers=False
)
plt.title("C3 Epsilon-APE Boxplot (No y-limit)")
plt.ylabel("Epsilon APE (%)")
plt.xlabel("Test period")
plt.legend(title="")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 4) Optional: test-period MAPE pivot table for report
# ============================================

def period_metric_pivot(metrics_by_period: pd.DataFrame, metric_col: str):
    return (
        metrics_by_period
        .pivot(index="period", columns="method", values=metric_col)
        .reset_index()
        .sort_values("period")
    )

print("C1 period MAPE (0-100):")
display(period_metric_pivot(art_c1.metrics_by_period, "MAPE_0_100"))

print("C3 period MAPE (0-100):")
display(period_metric_pivot(art_c3.metrics_by_period, "MAPE_0_100"))

# ============================================
# 5) Optional: load saved prediction parquet files
# ============================================
c1_pred = pd.read_parquet("forecasting/c1_prediction.parquet")
c3_pred = pd.read_parquet("forecasting/c3_prediction.parquet")

print("c1_pred shape:", c1_pred.shape)
print("c3_pred shape:", c3_pred.shape)

display(c1_pred.head())
display(c3_pred.head())

# ============================================
# 6) Optional: actual vs prediction time-series (cluster aggregate, test only)
# ============================================

def plot_cluster_agg_ts(pred_df: pd.DataFrame, baseline_col: str, model_col: str, title: str):
    agg = (
        pred_df.groupby("date", as_index=False)[["y", baseline_col, model_col]]
        .sum()
        .sort_values("date")
    )

    plt.figure(figsize=(14, 5))
    plt.plot(agg["date"], agg["y"], label="Actual", linewidth=2)
    plt.plot(agg["date"], agg[baseline_col], label=f"Baseline ({baseline_col})", alpha=0.9)
    plt.plot(agg["date"], agg[model_col], label=f"Model ({model_col})", alpha=0.9)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Cluster daily total_sales")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_cluster_agg_ts(
    art_c1.pred_df,
    baseline_col="pred_snaive7",
    model_col="pred_two_stage",
    title="C1 Test Aggregate: Actual vs Predictions"
)

# For C3 baseline, use selected baseline name
c3_baseline_col_map = {
    "tsb": "pred_tsb",
    "sba": "pred_sba",
    "adida": "pred_adida",
}
c3_base_col = c3_baseline_col_map[art_c3.selected_baseline_name]

plot_cluster_agg_ts(
    art_c3.pred_df,
    baseline_col=c3_base_col,
    model_col="pred_two_stage",
    title=f"C3 Test Aggregate: Actual vs Predictions (baseline={art_c3.selected_baseline_name})"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_random_skus_train_test(
    art,
    cluster_label="C1",
    baseline_col="pred_snaive7",
    model_col="pred_two_stage",
    n_skus=3,
    random_state=123,
):
    rng = np.random.default_rng(random_state)

    train_df = art.train_panel.copy()   # columns: date, product_family_name, total_sales
    test_df = art.pred_df.copy()        # columns: date, product_family_name, y, baseline/model preds

    sku_list = test_df["product_family_name"].dropna().unique()
    n_pick = min(n_skus, len(sku_list))
    picked = rng.choice(sku_list, size=n_pick, replace=False)

    ncols = 1
    nrows = int(np.ceil(n_pick / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharex=False)
    axes = np.array(axes).reshape(-1)

    for i, sku in enumerate(picked):
        ax = axes[i]

        tr = train_df[train_df["product_family_name"] == sku].sort_values("date")
        te = test_df[test_df["product_family_name"] == sku].sort_values("date")

        # Training: only actual
        ax.plot(tr["date"], tr["total_sales"], color="gray", lw=1.6, label="Train Actual")

        # Testing: actual + baseline + model
        ax.plot(te["date"], te["y"], lw=2.0, label="Test Actual")
        ax.plot(te["date"], te[baseline_col], lw=1.6, label=f"Test Baseline ({baseline_col})")
        ax.plot(te["date"], te[model_col], lw=1.6, label=f"Test Model ({model_col})")

        # Mark train/test split
        if len(te) > 0:
            split_date = te["date"].min()
            ax.axvline(split_date, color="black", linestyle="--", lw=1)
            ax.text(split_date, ax.get_ylim()[1] * 0.95, "Test start", fontsize=8, va="top")

        ax.set_title(f"{cluster_label} | {sku}")
        ax.set_xlabel("Date")
        ax.set_ylabel("Daily sales")
        ax.legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

# C1
plot_random_skus_train_test(
    art_c1,
    cluster_label="C1",
    baseline_col="pred_snaive7",
    model_col="pred_two_stage",
    n_skus=3
)

# C3 (baseline is dynamic)
c3_baseline_col = f"pred_{art_c3.selected_baseline_name}"
plot_random_skus_train_test(
    art_c3,
    cluster_label="C3",
    baseline_col=c3_baseline_col,
    model_col="pred_two_stage",
    n_skus=3
)


In [ ]:
import c1c3_analysis
importlib.reload(c1c3_analysis)
from c1c3_analysis import run_cluster_analysis

# C1 analysis (do not save files)
c1_results = run_cluster_analysis(
    art=art_c1,
    cluster_tag="c1",
    baseline_col="pred_snaive7",
    model_col="pred_two_stage",
    rolling_metric="wmape",   # or "mape"
    rolling_window=7,
    eps=1.0,
    n_periods=4,
    save_svg=True,           # set True to save svg
    output_root="images",
)

display(c1_results["metric_table"])
display(c1_results["confusion_by_period"])
display(c1_results["error_decomposition"])
display(c1_results["distribution_shift_stats"])

In [ ]:
# C3 analysis (baseline inferred from selected baseline)
c3_baseline_col = f"pred_{art_c3.selected_baseline_name}"

c3_results = run_cluster_analysis(
    art=art_c3,
    cluster_tag="c3",
    baseline_col=c3_baseline_col,
    model_col="pred_two_stage",
    rolling_metric="wmape",
    rolling_window=7,
    eps=1.0,
    n_periods=4,
    save_svg=True,           # set True to save svg to images/c3_results
    output_root="images",
)

display(c3_results["metric_table"])
display(c3_results["confusion_by_period"])
display(c3_results["error_decomposition"])
display(c3_results["distribution_shift_stats"])